# Weibull analysis of observed failure times under PHC and APHC

This notebook fits Weibull models to the 12 failure times in the ReliaSoft two-stress example. It illustrates progressive hybrid censoring (PHC) and adaptive progressive hybrid censoring (APHC) using those recorded lifetimes.

**Run All** to generate a Word report, its PDF, and an Excel workbook in `observed_data_results` under the notebook's working directory. The data and report layout are included in the notebook.

PDF export uses LibreOffice when it is available and a Python renderer otherwise. The notebook installs the Python package ReportLab on first use if needed. Both renderers use the same report content and tables. Font metrics and line wrapping can vary slightly.

In [1]:
import os
import shutil
import subprocess
import sys
from importlib.util import find_spec
from pathlib import Path
from tempfile import TemporaryDirectory

packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "scipy": "scipy",
    "docx": "python-docx",
    "xlsxwriter": "XlsxWriter",
}
missing = [package for module, package in packages.items() if find_spec(module) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", *missing], check=True)

OUTPUT_DIR = Path("observed_data_results")
REPORT_FILENAME = "PHC_APHC_Observed_Data_Methods_and_Results"
WORKBOOK_FILENAME = "PHC_APHC_Observed_Data_Results.xlsx"
SOURCE_URL = "https://help.reliasoft.com/weibull25/content/alta_two-stress_example.htm"
INTERVAL_REFERENCE = "https://doi.org/10.1243/1748006XJRR247"

## 1 Observed failure times

The [ReliaSoft Two-Stress Example](https://help.reliasoft.com/weibull25/content/alta_two-stress_example.htm) gives four exact failure times at each of three stress settings. Temperature is in kelvin, voltage in volts, and lifetime in hours.

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.special import logsumexp

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

observations = pd.DataFrame(
    [
        ("O1", 348, 3, (620, 632, 658, 822)),
        ("O2", 378, 3, (216, 246, 332, 400)),
        ("O3", 378, 5, (380, 416, 460, 596)),
    ],
    columns=["Cell", "Temperature (K)", "Voltage (V)", "Lifetimes"],
).set_index("Cell")

observed_display = observations.drop(columns="Lifetimes").assign(
    **{"Observed lifetimes (h)": observations["Lifetimes"].map(
        lambda sample: ", ".join(map(str, sample))
    )}
)
display(observed_display)

,Temperature (K),Voltage (V),Observed lifetimes (h)
Cell,,,
O1,348,3,"620, 632, 658, 822"
O2,378,3,"216, 246, 332, 400"
O3,378,5,"380, 416, 460, 596"


## 2 Fit the complete-data Weibull models

Estimate shape and scale by maximum likelihood separately at each stress setting. Exact failures contribute their failure density and right-censored units contribute their survival probability. For a trial shape, compute the best-fitting scale from the failure and censoring times, then solve the profile score for the shape. Use the fitted parameters to compute the median lifetime, B50.

In [3]:
def fit_weibull(failure_times, censor_times=()):
    """Return a shape and scale for each independent row of failure times."""
    failures = np.atleast_2d(np.asarray(failure_times, dtype=float))
    censored = np.asarray(censor_times, dtype=float).reshape(-1)
    if failures.ndim != 2 or failures.shape[1] == 0:
        raise ValueError("Each sample needs an observed failure.")

    n_rows, n_failures = failures.shape
    times = np.column_stack(
        (failures, np.broadcast_to(censored, (n_rows, censored.size)))
    )
    if np.any(~np.isfinite(times)) or np.any(times <= 0):
        raise ValueError("All event and censoring times must be positive and finite.")

    # Normalize log times within each sample to keep the shape search stable.
    log_times = np.log(times)
    log_min = log_times.min(axis=1)
    log_span = np.ptp(log_times, axis=1)
    if np.any(log_span <= 0):
        raise ValueError("A finite Weibull fit needs distinct times.")

    normalized = (log_times - log_min[:, None]) / log_span[:, None]
    mean_failure_log = normalized[:, :n_failures].mean(axis=1)
    if np.any(mean_failure_log >= 1):
        raise ValueError("At least one failure must precede the largest recorded time.")

    # The scaled shape score is decreasing through its root.
    def shape_score(scaled_shape):
        weights = np.exp(scaled_shape[:, None] * (normalized - 1.0))
        weighted_mean = (weights * normalized).sum(axis=1) / weights.sum(axis=1)
        return 1.0 / scaled_shape + mean_failure_log - weighted_mean

    lower = np.zeros(n_rows)
    upper = np.full(n_rows, 16.0)
    for _ in range(60):
        expand = shape_score(upper) > 0
        if not expand.any():
            break
        upper[expand] *= 2.0
    else:
        raise RuntimeError("Unable to bracket the Weibull shape estimate.")

    for _ in range(48):
        midpoint = (lower + upper) / 2.0
        above_root = shape_score(midpoint) <= 0
        upper = np.where(above_root, midpoint, upper)
        lower = np.where(above_root, lower, midpoint)

    scaled_shape = (lower + upper) / 2.0
    shape = scaled_shape / log_span
    log_power_sum = logsumexp(scaled_shape[:, None] * normalized, axis=1)
    log_scale = log_min + (log_power_sum - np.log(n_failures)) / shape
    scale = np.exp(log_scale)
    return shape, scale


def weibull_median(shape, scale):
    return scale * np.log(2.0) ** (1.0 / shape)


complete_rows = []
for cell, row in observations.iterrows():
    shape, scale = (values.item() for values in fit_weibull(row["Lifetimes"]))
    complete_rows.append({
        "Cell": cell,
        "Temperature (K)": row["Temperature (K)"],
        "Voltage (V)": row["Voltage (V)"],
        "Failures": len(row["Lifetimes"]),
        "Shape": shape,
        "Scale (h)": scale,
        "B50 (h)": weibull_median(shape, scale),
    })

complete_fits = pd.DataFrame(complete_rows).set_index("Cell")

## 3 Estimate uncertainty in the complete-data medians

At each setting, draw 200,000 samples of four lifetimes from its fitted Weibull model and refit each sample. Multiply the log ratio of the bootstrap median to the original fitted median by the bootstrap shape estimate. Invert the 2.5th and 97.5th percentiles of this pivotal statistic to obtain the 95% confidence limits. One seeded random number stream runs through O1, O2 and O3 in order.

In [4]:
N_BOOTSTRAP = 200_000
BOOTSTRAP_SEED = 531211
rng = np.random.default_rng(BOOTSTRAP_SEED)

median_intervals = []
bootstrap_rows = []
for cell, fitted in complete_fits.iterrows():
    sample_size = len(observations.at[cell, "Lifetimes"])
    draws = fitted["Scale (h)"] * rng.weibull(
        fitted["Shape"],
        size=(N_BOOTSTRAP, sample_size),
    )
    boot_shape, boot_scale = fit_weibull(draws)
    boot_median = weibull_median(boot_shape, boot_scale)

    reference_median = fitted["B50 (h)"]
    pivot = boot_shape * np.log(boot_median / reference_median)
    low_pivot, high_pivot = np.quantile(pivot, [0.025, 0.975])
    lower = reference_median * np.exp(-high_pivot / fitted["Shape"])
    upper = reference_median * np.exp(-low_pivot / fitted["Shape"])
    median_intervals.append((cell, lower, upper))
    bootstrap_rows.append((cell, sample_size, N_BOOTSTRAP, BOOTSTRAP_SEED, low_pivot, high_pivot))

median_intervals = pd.DataFrame(
    median_intervals,
    columns=["Cell", "95% CI lower (h)", "95% CI upper (h)"],
).set_index("Cell")
complete_results = complete_fits.join(median_intervals)

bootstrap_details = pd.DataFrame(
    bootstrap_rows,
    columns=["Cell", "Sample size", "Repetitions", "Seed", "Pivot 2.5%", "Pivot 97.5%"],
)

display(complete_results.round({
    "Shape": 3,
    "Scale (h)": 1,
    "B50 (h)": 1,
    "95% CI lower (h)": 1,
    "95% CI upper (h)": 1,
}))

,Temperature (K),Voltage (V),Failures,Shape,Scale (h),B50 (h),95% CI lower (h),95% CI upper (h)
Cell,,,,,,,,
O1,348,3,4,8.149,721.4,689.7,497.9,856.9
O2,378,3,4,4.605,327.3,302.3,170.5,444.9
O3,378,5,4,5.811,498.6,468.2,297.4,635.7


## 4 Apply PHC and APHC to the recorded lifetimes

Each setting starts with four units, targets three failures, and schedules withdrawals of `(0, 0, 1)`. The threshold at each setting is its fitted complete-data median. The exercise is retrospective because the same recorded lifetimes supply the thresholds.

- **PHC** ends at the third failure if it occurs strictly before the threshold. Otherwise the test stops at the threshold. All surviving units are right-censored at the stop.
- **APHC** runs to the third failure and right-censors the remaining unit at that time. This illustration schedules all withdrawals at the stop.

Refit the Weibull model using the failure times and one censoring time per censored unit.

In [5]:
FAILURE_TARGET = 3


def apply_censoring(lifetimes, threshold, scheme, target):
    """Apply a plan with every withdrawal at the stop."""
    times = np.sort(np.asarray(lifetimes, dtype=float))
    if not 1 <= target < times.size:
        raise ValueError("The target must be smaller than the number of units.")

    potential_failures = times[:target]
    if scheme == "PHC":
        failures = potential_failures[potential_failures < threshold]
        reached_target = failures.size == target
        stop = failures[-1] if reached_target else threshold
    elif scheme == "APHC":
        failures = potential_failures
        reached_target = True
        stop = failures[-1]
    else:
        raise ValueError(f"Unknown censoring scheme {scheme}")

    censored = np.full(times.size - failures.size, stop)
    return failures, censored, stop, reached_target


censoring_rows = []
censored_fit_rows = []
for cell, row in observations.iterrows():
    threshold = complete_fits.at[cell, "B50 (h)"]
    for scheme in ("PHC", "APHC"):
        failures, censored, stop, reached_target = apply_censoring(
            row["Lifetimes"], threshold, scheme, FAILURE_TARGET
        )
        censoring_rows.append({
            "Cell": cell,
            "Scheme": scheme,
            "Threshold (h)": threshold,
            "Recorded failure times (h)": ", ".join(f"{x:g}" for x in failures),
            "Failures": failures.size,
            "Right-censored units": censored.size,
            "Stop and censoring time (h)": stop,
            "Stop reason": "Failure target" if reached_target else "Time threshold",
        })
        shape, scale = (values.item() for values in fit_weibull(failures, censored))
        censored_fit_rows.append({
            "Cell": cell,
            "Scheme": scheme,
            "Failures": failures.size,
            "Shape": shape,
            "Scale (h)": scale,
            "B50 (h)": weibull_median(shape, scale),
        })

censoring_records = pd.DataFrame(censoring_rows).set_index(["Cell", "Scheme"])
censored_fits = pd.DataFrame(censored_fit_rows).set_index(["Cell", "Scheme"])

display(censoring_records.round({"Threshold (h)": 1, "Stop and censoring time (h)": 1}))
display(censored_fits.round({"Shape": 3, "Scale (h)": 1, "B50 (h)": 1}))

Threshold (h) Recorded failure times (h)  Failures  Right-censored units  Stop and censoring time (h)     Stop reason
Cell Scheme                                                                                                                       
O1   PHC             689.7              620, 632, 658         3                     1                        658.0  Failure target
     APHC            689.7              620, 632, 658         3                     1                        658.0  Failure target
O2   PHC             302.3                   216, 246         2                     2                        302.3  Time threshold
     APHC            302.3              216, 246, 332         3                     1                        332.0  Failure target
O3   PHC             468.2              380, 416, 460         3                     1                        460.0  Failure target
     APHC            468.2              380, 416, 460         3                     1                        460.0  Failure target

Failures   Shape  Scale (h)  B50 (h)
Cell Scheme                                      
O1   PHC            3  37.464      653.6    647.2
     APHC           3  37.464      653.6    647.2
O2   PHC            2   4.677      319.5    295.4
     APHC           3   5.126      315.9    294.1
O3   PHC            3  12.762      451.5    438.7
     APHC           3  12.762      451.5    438.7

## 5 Estimate the stress coefficients

Regress the three estimated scales and the three estimated shapes separately on an intercept, reciprocal temperature, and natural log of voltage. Center and scale the stress predictors for the solve, then return the coefficients to their original units. Each equation has three settings and three coefficients. The fitted cell values match the inputs exactly and leave zero residual degrees of freedom.

In [6]:
predictors = np.column_stack((
    1.0 / observations["Temperature (K)"],
    np.log(observations["Voltage (V)"]),
))
predictor_means = predictors.mean(axis=0)
predictor_sds = predictors.std(axis=0)
standardized_design = np.column_stack((
    np.ones(len(observations)),
    (predictors - predictor_means) / predictor_sds,
))

fit_tables = {
    "Complete data": complete_fits,
    "PHC": censored_fits.xs("PHC", level="Scheme"),
    "APHC": censored_fits.xs("APHC", level="Scheme"),
}
stress_estimates = {}
for name, fits in fit_tables.items():
    responses = fits.loc[observations.index, ["Scale (h)", "Shape"]].to_numpy()
    standardized = np.linalg.solve(standardized_design, responses)
    slopes = standardized[1:] / predictor_sds[:, None]
    intercepts = standardized[0] - predictor_means @ slopes
    coefficients = np.vstack((intercepts, slopes))
    stress_estimates[name] = coefficients.T.ravel()

stress_coefficients = pd.DataFrame(
    stress_estimates,
    index=pd.Index(["a0", "a1", "a2", "c0", "c1", "c2"], name="Coefficient"),
)
display(stress_coefficients.round(4))

,Complete data,PHC,APHC
Coefficient,,,
a0,-4.612407e+03,-3.840167e+03,-3.892521e+03
a1,1.727971e+06,1.464995e+06,1.480557e+06
a2,3.353329e+02,2.584873e+02,2.654353e+02
c0,-3.909730e+01,-3.930359e+02,-3.864118e+02
c1,1.553924e+04,1.437627e+05,1.417937e+05
c2,2.360700e+00,1.582730e+01,1.494820e+01


## 6 Prepare the result tables

Create the tables for the report and workbook from the fitted values. The Word report includes the seven tables in the reference layout. Excel retains full-precision numeric results and the original failure time for each unit.

In [7]:
observed_units = [
    (cell, row["Temperature (K)"], row["Voltage (V)"], unit, time)
    for cell, row in observations.iterrows()
    for unit, time in enumerate(row["Lifetimes"], start=1)
]
observed_export = pd.DataFrame(
    observed_units,
    columns=["Cell", "Temperature (K)", "Voltage (V)", "Unit", "Original failure (h)"],
)

complete_export = complete_results.reset_index()
censored_export = censored_fits.reset_index()
complete_medians = censored_export["Cell"].map(complete_fits["B50 (h)"])
censored_export["Median change (h)"] = censored_export["B50 (h)"] - complete_medians
censoring_export = censoring_records.reset_index()

# Unit numbers follow the original order of the source observations in each cell.
unit_records = []
for (cell, scheme), record in censoring_records.iterrows():
    lifetimes = np.asarray(observations.at[cell, "Lifetimes"], dtype=float)
    chronological_units = np.argsort(lifetimes, kind="stable")
    failure_units = set(chronological_units[: int(record["Failures"])])
    for index, time in enumerate(lifetimes):
        failed = index in failure_units
        analysis_time = time if failed else record["Stop and censoring time (h)"]
        unit_records.append((
            cell, scheme, index + 1, time, analysis_time,
            "Failure" if failed else "Right-censored",
        ))

censoring_unit_export = pd.DataFrame(
    unit_records,
    columns=["Cell", "Scheme", "Unit", "Original failure (h)", "Analysis time (h)", "Record type"],
)

coefficient_export = stress_coefficients.reset_index()
coefficient_export.insert(1, "Response", ["Scale"] * 3 + ["Shape"] * 3)
coefficient_export.insert(2, "Predictor", ["Intercept", "Inverse temperature", "Log voltage"] * 2)

# Verify that the exported unit records reproduce the fitted samples.
for (cell, scheme), units in censoring_unit_export.groupby(["Cell", "Scheme"], sort=False):
    retained = units["Record type"].eq("Failure")
    summary = censoring_records.loc[cell, scheme]
    assert len(units) == len(observations.at[cell, "Lifetimes"])
    assert retained.sum() == summary["Failures"]
    assert (~retained).sum() == summary["Right-censored units"]
    assert (units.loc[~retained, "Analysis time (h)"] == summary["Stop and censoring time (h)"]).all()
    assert (units["Analysis time (h)"] <= units["Original failure (h)"]).all()

## 7 Build the Word report

In [8]:
from docx import Document
from docx.enum.style import WD_STYLE_TYPE
from docx.enum.table import WD_CELL_VERTICAL_ALIGNMENT, WD_TABLE_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.opc.constants import RELATIONSHIP_TYPE as RT
from docx.shared import Inches, Pt, RGBColor, Twips

REPORT_TITLE = "Observed Failure Times Under PHC and APHC"
REPORT_FONT = "Times New Roman"
REPORT_WIDTH = 9994


def word_element(name, **attrs):
    element = OxmlElement(f"w:{name}")
    for key, value in attrs.items():
        element.set(qn(f"w:{key}"), str(value))
    return element


def new_report():
    doc = Document()
    doc.core_properties.title = REPORT_TITLE
    doc.core_properties.subject = "Weibull accelerated life testing"
    doc.core_properties.author = "Ramnivas Jat"
    section = doc.sections[0]
    section.page_width, section.page_height = Inches(8.5), Inches(11)
    section.left_margin = section.right_margin = Inches(0.78)
    section.top_margin, section.bottom_margin = Inches(0.70), Inches(0.67)
    section.footer_distance = Inches(0.32)
    footer = section.footer.paragraphs[0]
    footer.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    footer._p.append(word_element("fldSimple", instr="PAGE"))

    styles = doc.styles
    # name, size, bold, points before, points after, keep with next
    formats = [
        ("Normal", 11.5, False, 0, 6, False),
        ("Title", 21, True, 0, 11, True),
        ("Heading 1", 15, True, 9, 6, True),
        ("Heading 2", 12, True, 7, 4, True),
        ("Table Caption", 10.5, True, 7, 4, True),
        ("Table Note", 10, False, 3, 7, False),
        ("Table Entry", 10, False, 0, 0, False),
        ("Censoring Step", 11.5, False, 0, 5, False),
        ("Reference Entry", 10.5, False, 0, 5, False),
        ("Footer", 11.5, False, 0, 0, False),
    ]
    for name, size, bold, before, after, keep in formats:
        if name not in styles:
            styles.add_style(name, WD_STYLE_TYPE.PARAGRAPH).base_style = styles["Normal"]
        style = styles[name]
        style.font.name = REPORT_FONT
        style.font.size = Pt(size)
        style.font.bold = bold
        style.font.color.rgb = RGBColor(0, 0, 0)
        fonts = style.element.get_or_add_rPr().rFonts
        for attr in list(fonts.attrib):
            if attr.lower().endswith("theme"):
                del fonts.attrib[attr]
        for attr in ("ascii", "hAnsi", "eastAsia", "cs"):
            fonts.set(qn(f"w:{attr}"), REPORT_FONT)
        fmt = style.paragraph_format
        fmt.space_before, fmt.space_after = Pt(before), Pt(after)
        fmt.keep_with_next, fmt.keep_together = keep, True
        fmt.widow_control = True
        fmt.line_spacing = 1.0 if name in {"Table Entry", "Footer"} else 1.1
        border = style.element.find(".//" + qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)
    styles["Title"].paragraph_format.line_spacing = 1.02
    for name in ("Censoring Step", "Reference Entry"):
        fmt = styles[name].paragraph_format
        fmt.left_indent, fmt.first_line_indent = Pt(15.85), Pt(-15.85)

    doc.add_paragraph(REPORT_TITLE, style="Title")
    return doc


def word_number(value, decimals=1):
    return f"{value:,.{decimals}f}".replace("-", "−")


def word_table(doc, caption, frame, widths, *, headers=None, decimals=None, compact=False):
    headers = list(frame.columns) if headers is None else headers
    decimals = {} if decimals is None else decimals
    assert len(headers) == len(widths) == len(frame.columns)
    assert abs(sum(widths) - REPORT_WIDTH) <= 3
    doc.add_paragraph(caption, style="Table Caption")
    table = doc.add_table(rows=1, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    properties = table._tbl.tblPr
    total_width = properties.find(qn("w:tblW"))
    total_width.set(qn("w:type"), "dxa")
    total_width.set(qn("w:w"), str(REPORT_WIDTH))
    borders = word_element("tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        borders.append(word_element(edge, val="single", sz=4, color="D9D9D9"))
    properties.append(borders)
    margins = word_element("tblCellMar")
    for side, width in (("top", 39), ("bottom", 39), ("left", 61), ("right", 61)):
        margins.append(word_element(side, w=width, type="dxa"))
    properties.append(margins)
    for column, width in zip(table.columns, widths, strict=True):
        column.width = Twips(width)

    numeric = [pd.api.types.is_numeric_dtype(frame[column]) for column in frame]
    for index, values in enumerate([headers, *frame.itertuples(index=False, name=None)]):
        is_header = index == 0
        row = table.rows[0] if is_header else table.add_row()
        row_properties = row._tr.get_or_add_trPr()
        row_properties.append(word_element("cantSplit"))
        if is_header:
            row_properties.append(word_element("tblHeader"))
        for i, (cell, value, width) in enumerate(zip(row.cells, values, widths, strict=True)):
            cell.width = Twips(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            paragraph = cell.paragraphs[0]
            paragraph.style = "Table Entry"
            paragraph.paragraph_format.keep_with_next = is_header
            if numeric[i]:
                paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER if is_header else WD_ALIGN_PARAGRAPH.RIGHT
            if is_header:
                cell._tc.get_or_add_tcPr().append(word_element("shd", fill="E8E8E8"))
            text = word_number(value, decimals[i]) if not is_header and i in decimals else str(value)
            run = paragraph.add_run(text)
            run.bold = is_header
            run.font.size = Pt(9.5 if compact else 10)
    return table


def word_heading(doc, text, level, *, page=False):
    paragraph = doc.add_heading(text, level)
    if page:
        paragraph.paragraph_format.page_break_before = True
        paragraph.paragraph_format.space_before = Pt(0)


def word_step(doc, label, text):
    paragraph = doc.add_paragraph(style="Censoring Step")
    paragraph.add_run(f"{label} ").bold = True
    paragraph.add_run(text)


def word_reference(doc, text, label, url):
    paragraph = doc.add_paragraph(text + " ", style="Reference Entry")
    link = word_element("hyperlink")
    link.set(qn("r:id"), paragraph.part.relate_to(url, RT.HYPERLINK, is_external=True))
    run = word_element("r")
    props = word_element("rPr")
    props.append(word_element("u", val="single"))
    run.append(props)
    content = word_element("t")
    content.text = label
    run.append(content)
    link.append(run)
    paragraph._p.append(link)

In [9]:
def write_word_report(path):
    doc = new_report()
    paragraph = doc.add_paragraph
    paragraph(
        "We analysed 12 failure times from the ReliaSoft two-stress example using the Weibull "
        "accelerated life testing (ALT) model. We fitted the complete data first, then applied "
        "progressive hybrid censoring (PHC) and adaptive progressive hybrid censoring (APHC) "
        "to the same lifetimes. The comparison shows how the two stopping rules changed the "
        "records and the resulting estimates."
    )

    word_heading(doc, "Approach", 1)
    word_heading(doc, "1 Group the observed failures by stress setting", 2)
    paragraph(
        "The example reports four exact failure times at each of three temperature and voltage "
        "settings (ReliaSoft, 2025). We refer to each setting as a cell and label them O1, O2, "
        "and O3. The lifetimes are in hours."
    )
    word_table(
        doc, "Table 1. Observed failure times", observed_display.reset_index(),
        [1079, 2221, 1814, 4880],
        headers=["Cell", "Temperature (K)", "Voltage (V)", "Failure times (h)"],
    )
    word_heading(doc, "2 Fit a Weibull model to the complete data", 2)
    paragraph(
        "We fitted one two-parameter Weibull model to the four failure times in each cell, "
        "treating the lifetimes as independent. Shape describes how the failure rate changes "
        "over time. Scale sets the time scale in hours."
    )
    paragraph(
        "We chose the shape and scale that gave the highest likelihood for the four exact "
        "failures. Step 5 describes how we found the estimates. We then used the fitted model "
        "to find the time by which half the units would be expected to fail. This is the fitted "
        "median lifetime. The complete-data estimates provide a reference for the PHC and APHC fits."
    )
    word_heading(doc, "3 Estimate uncertainty in the median lifetime", 2)
    paragraph(
        "We used a Weibull pivotal bootstrap to calculate a 95% confidence interval for the "
        "complete-data median at each cell (Phan and McCool, 2009)."
    )
    word_step(
        doc, "1.", f"Draw {N_BOOTSTRAP:,} samples from the Weibull model fitted to that cell. "
        "Each sample contains four lifetimes, matching the size of the observed sample."
    )
    word_step(
        doc, "2.", "Refit the model to each sample and calculate its median. Divide that median "
        "by the median fitted to the observed data. Take the natural log of the ratio and "
        "multiply it by the shape estimated from that bootstrap sample."
    )
    word_step(
        doc, "3.", "Find the 2.5th and 97.5th percentiles of these values. For each percentile, "
        "reverse its sign, divide by the shape fitted to the observed data, and apply the "
        "exponential function. Multiply the result by the original fitted median to get a "
        "limit in hours. The higher percentile gives the lower limit."
    )
    paragraph("The interval describes uncertainty in the complete-data median under the Weibull model.")

    word_heading(doc, "4 Apply the censoring rules to the observed lifetimes", 2, page=True)
    paragraph(
        "Both schemes start with four units and aim to record three failures. We leave all "
        "survivors on test after the first two failures. If the test reaches the third failure, "
        "we withdraw the remaining unit at that time. Each cell uses its complete-data median "
        "lifetime as the time threshold."
    )
    word_step(
        doc, "PHC.", "If the third failure occurs before the threshold, stop there and "
        "right-censor the last unit. Otherwise, stop at the threshold, retain the failures "
        "strictly before it, and right-censor every unit still on test at that time."
    )
    word_step(
        doc, "APHC.", "Continue to the third failure, even if the time threshold has already "
        "passed. Record the first three failures and right-censor the fourth unit at the "
        "third failure time. This plan schedules all withdrawals at the stop."
    )
    paragraph(
        "A right-censored record tells us the unit survived to the time the test stopped. "
        "We applied both rules retrospectively using the recorded lifetimes, which also "
        "supplied the estimated thresholds."
    )
    word_heading(doc, "5 Refit the Weibull model using the retained records", 2)
    paragraph(
        "The Weibull fit uses the failure density for each exact failure and the survival "
        "probability for each right-censored unit. Each unit contributes one record, even "
        "when several units are censored at the same time."
    )
    word_step(
        doc, "1.", "For a trial shape, raise each failure and censoring time to that power. "
        "Add the values and divide by the number of exact failures. Raise this result to the "
        "reciprocal of the trial shape to find its best-fitting scale."
    )
    word_step(
        doc, "2.", "Find the shape that gives the highest likelihood, using the best scale at "
        "each trial shape. We rescaled the log times for numerical stability and narrowed "
        "the search range by repeatedly halving it."
    )
    word_step(
        doc, "3.", "Use the selected shape and scale to calculate the median. Repeat the fit "
        "for both schemes at each cell. The complete-data and bootstrap fits follow the "
        "same procedure with four exact failures in each sample."
    )
    word_heading(doc, "6 Estimate the stress coefficients", 2)
    paragraph(
        "We ran two least-squares regressions across the three cells for each set of Weibull "
        "fits (complete data, PHC, and APHC). One used the fitted scales as its response and "
        "the other used the fitted shapes. Both used an intercept, the reciprocal of "
        "temperature measured in kelvin, and the natural log of voltage measured in volts."
    )
    paragraph(
        "We subtracted each stress predictor’s mean and divided by its population standard "
        "deviation before fitting. We then converted the slopes and intercept back to the "
        "original predictor units."
    )
    paragraph(
        "Three stress settings determine three coefficients in each regression. The fitted "
        "values match the three cell estimates exactly, leaving zero residual degrees of "
        "freedom. The results table therefore reports point estimates."
    )

    word_heading(doc, "Results", 1, page=True)
    word_heading(doc, "Complete data Weibull estimates", 2)
    complete_columns = [
        "Cell", "Failures", "Shape", "Scale (h)", "B50 (h)",
        "95% CI lower (h)", "95% CI upper (h)",
    ]
    word_table(
        doc, "Table 2. Fitted shape, scale, and median with 95% confidence limits",
        complete_export[complete_columns], [986, 1155, 1309, 1386, 1524, 1817, 1817],
        headers=[
            "Cell", "Failures", "Shape", "Scale (h)", "Median (h)",
            "Lower 95%\nlimit (h)", "Upper 95%\nlimit (h)",
        ],
        decimals={2: 3, 3: 1, 4: 1, 5: 1, 6: 1},
    )
    medians = complete_results["B50 (h)"]
    paragraph(
        f"{medians.idxmax()} has the longest fitted median at {medians.max():.1f} hours. "
        f"{medians.idxmin()} has the shortest at {medians.min():.1f} hours. With four "
        "observations per cell, the confidence limits cover a broad range of median lifetimes."
    )
    word_heading(doc, "Records that enter the censored fits", 2)
    word_table(
        doc, "Table 3. Retained failures and censoring by stopping rule", censoring_export,
        [764, 1024, 1146, 2170, 993, 1192, 1146, 1559],
        headers=[
            "Cell", "Scheme", "Threshold\n(h)", "Retained failure times (h)", "Failures",
            "Censored\nunits", "Stop time\n(h)", "Stop reason",
        ],
        decimals={2: 1, 6: 1}, compact=True,
    )
    paragraph(
        "Each censored unit is recorded at the stop time shown. The fits use the unrounded times.",
        style="Table Note",
    )
    o2_phc = censoring_records.loc["O2", "PHC"]
    o2_aphc = censoring_records.loc["O2", "APHC"]
    o2_threshold = o2_phc["Threshold (h)"]
    o2_second = observations.at["O2", "Lifetimes"][1]
    o2_third = o2_aphc["Stop and censoring time (h)"]
    paragraph(
        f"O2 shows where the rules differ. Its threshold of {o2_threshold:.1f} hours falls "
        f"between failure times {o2_second:g} and {o2_third:g} hours. PHC stops there with "
        "two failures and two censored units. APHC continues to "
        f"{o2_third:g} hours, records a third failure, and censors the fourth unit at that time."
    )
    word_heading(doc, "Weibull estimates after censoring", 2)
    word_table(
        doc, "Table 4. Censored Weibull fits and changes in median lifetime", censored_export,
        [953, 1292, 1211, 1518, 1614, 1614, 1792],
        headers=["Cell", "Scheme", "Failures", "Shape", "Scale (h)", "Median (h)", "Median change\n(h)"],
        decimals={3: 3, 4: 1, 5: 1, 6: 1},
    )
    paragraph(
        "Median change is the censored-fit median minus the complete-data median for the "
        "same cell, calculated before rounding.",
        style="Table Note",
    )

    word_heading(doc, "Stress coefficients and interpretation", 2, page=True)
    subscript_digits = str.maketrans("012", "₀₁₂")
    word_table(
        doc, "Table 5. Stress coefficient estimates for the three sets of fits",
        coefficient_export.assign(Coefficient=coefficient_export["Coefficient"].str.translate(subscript_digits)),
        [786, 1112, 2135, 2105, 1928, 1928],
        headers=["Term", "Response", "Predictor", "Complete data", "PHC", "APHC"],
        decimals={3: 4, 4: 4, 5: 4},
    )
    paragraph("The a coefficients describe scale. The c coefficients describe shape.", style="Table Note")
    word_heading(doc, "What the comparison shows", 2)
    paragraph(
        "At O1 and O3, the third failure occurs before the threshold. Both schemes therefore "
        "give identical records and Weibull estimates. At O2, the fitted medians are "
        f"{censored_fits.at[('O2', 'PHC'), 'B50 (h)']:.1f} hours under PHC and "
        f"{censored_fits.at[('O2', 'APHC'), 'B50 (h)']:.1f} hours under APHC, compared "
        f"with the complete-data estimate of {medians.at['O2']:.1f} hours."
    )
    largest_shift = censored_export.loc[censored_export["Median change (h)"].abs().idxmax()]
    shift_cell = largest_shift["Cell"]
    paragraph(
        f"{shift_cell} has the largest median change in hours, a decrease of "
        f"{abs(largest_shift['Median change (h)']):.1f} hours. Its fitted shape also increases "
        f"from {complete_results.at[shift_cell, 'Shape']:.3f} to {largest_shift['Shape']:.3f} "
        "under both censoring schemes. Comparing both parameters shows how censoring changes "
        "the fitted distribution as well as its median."
    )
    paragraph(
        "All six censored-fit medians fall within the corresponding complete-data confidence "
        "limits. The fits share observations, so this comparison describes the size of the "
        "shifts in the fitted medians. The stress regressions describe the three observed settings."
    )
    word_heading(doc, "Bootstrap percentiles", 2)
    word_table(
        doc, "Table 6. Percentiles used to construct the median intervals", bootstrap_details,
        [1003, 1666, 1682, 1504, 2005, 2135],
        headers=[
            "Cell", "Units per draw", "Repetitions", "Initial seed",
            "2.5th percentile", "97.5th percentile",
        ],
        decimals={2: 0, 4: 5, 5: 5},
    )
    paragraph("The seed starts one random number stream used in cell order O1, O2, O3.", style="Table Note")

    word_heading(doc, "Appendix Unit records for the censored fits", 1, page=True)
    paragraph(
        "The analysis time is the exact failure time or the last time at which the unit "
        "was known to be alive. Unit numbers follow the order in Table 1."
    )
    word_table(
        doc, "Table A1. Records for all four units under each stopping rule", censoring_unit_export,
        [940, 1435, 891, 2259, 2045, 2424],
        headers=["Cell", "Scheme", "Unit", "Original failure\n(h)", "Analysis time\n(h)", "Record type"],
        decimals={3: 0, 4: 1},
    )
    word_heading(doc, "References", 2)
    word_reference(
        doc, "ReliaSoft. (2025). Two-Stress Example. Weibull++ 2025 Help.", "Source data", SOURCE_URL,
    )
    word_reference(
        doc,
        "Phan, L. D., and McCool, J. I. (2009). Exact confidence intervals for Weibull "
        "parameters and percentiles. Proceedings of the Institution of Mechanical Engineers, "
        "Part O: Journal of Risk and Reliability, 223(4), 387–394.",
        "DOI", INTERVAL_REFERENCE,
    )
    doc.save(path)

## 8 Build the Excel workbook

In [10]:
import xlsxwriter
from xlsxwriter.utility import xl_rowcol_to_cell


def write_excel_results(path):
    # The fitted parameters and unit-level records are the notebook results at full precision.
    sheets = [
        (
            "Complete fits", "Complete data Weibull estimates", complete_export,
            "The 95% confidence limits are for the complete-data median in each cell.",
            [10, 17, 13, 12, 13, 15, 15, 21, 21], None,
        ),
        (
            "Censored fits", "PHC and APHC Weibull estimates", censored_export,
            "Median change compares the censored fit with the complete-data fit in the same cell.",
            [10, 13, 12, 13, 15, 15, 22], None,
        ),
        (
            "Stress coefficients", "Temperature and voltage coefficients", coefficient_export,
            "The three observed settings give three point estimates per equation and zero residual degrees of freedom.",
            [15, 14, 25, 21, 21, 21], None,
        ),
        (
            "Censoring summary", "Records retained under each stopping rule", censoring_export,
            "The plan starts with four units, targets three failures, and schedules withdrawals of (0, 0, 1).",
            [10, 13, 16, 31, 12, 19, 27, 20], None,
        ),
        (
            "Censoring records", "Unit records used for censored Weibull fitting", censoring_unit_export,
            "Analysis time is the failure time or the time of right-censoring.",
            [10, 13, 10, 21, 19, 19], None,
        ),
        (
            "Bootstrap details", "Weibull pivotal bootstrap", bootstrap_details,
            "The initial seed starts one random number stream used in cell order O1, O2, O3.",
            [10, 16, 16, 15, 19, 19], (INTERVAL_REFERENCE, "Pivotal interval reference"),
        ),
        (
            "Observed lifetimes", "Original failure times and stress settings", observed_export,
            "The published example contains twelve exact failures.",
            [10, 20, 15, 12, 25], (SOURCE_URL, "ReliaSoft source"),
        ),
    ]
    header_row = 4
    first_result_row = header_row + 1
    complete_row = {cell: first_result_row + i for i, cell in enumerate(complete_export["Cell"])}
    integer_columns = {
        "Temperature (K)", "Voltage (V)", "Unit", "Failures", "Original failure (h)",
        "Right-censored units", "Sample size", "Repetitions", "Seed",
    }
    number_formats = {
        **{column: "0" for column in integer_columns},
        "Repetitions": "#,##0",
        "Shape": "0.000",
        "Complete data": "#,##0.0000",
        "PHC": "#,##0.0000",
        "APHC": "#,##0.0000",
        "Pivot 2.5%": "0.00000",
        "Pivot 97.5%": "0.00000",
    }

    with xlsxwriter.Workbook(path, {"strings_to_urls": False, "strings_to_formulas": False}) as book:
        book.set_properties({"title": REPORT_TITLE, "author": "Ramnivas Jat"})
        title = book.add_format({"bold": True, "font_size": 15, "font_color": "#203650"})
        note = book.add_format({"font_size": 10, "font_color": "#435365"})
        header = book.add_format({
            "bold": True, "font_color": "#FFFFFF", "bg_color": "#203650",
            "align": "center", "valign": "vcenter", "text_wrap": True,
            "right": 1, "right_color": "#FFFFFF",
        })
        body_formats = {}
        for name, heading, frame, description, widths, source in sheets:
            sheet = book.add_worksheet(name)
            sheet.hide_gridlines(2)
            sheet.freeze_panes(first_result_row, 0)
            sheet.set_row(1, 25)
            sheet.set_row(2, 17)
            sheet.set_row(header_row, 31)
            sheet.write_string(1, 0, heading, title)
            sheet.write_string(2, 0, description, note)
            if source:
                sheet.write_url(3, 0, source[0], string=source[1])
            if name in {"Complete fits", "Censored fits", "Stress coefficients"}:
                sheet.set_tab_color("#203650")

            for col, (label, width) in enumerate(zip(frame.columns, widths, strict=True)):
                sheet.set_column(col, col, width)
                sheet.write_string(header_row, col, label, header)

            for index, values in enumerate(frame.itertuples(index=False, name=None)):
                row = first_result_row + index
                sheet.set_row(row, 19)
                for col, (label, value) in enumerate(zip(frame.columns, values, strict=True)):
                    numeric = not isinstance(value, str)
                    num_format = number_formats.get(label, "#,##0.0") if numeric else None
                    style_key = (num_format, index % 2)
                    if style_key not in body_formats:
                        properties = {
                            "valign": "vcenter", "font_size": 11,
                            "align": "right" if numeric else "left", "indent": 1,
                        }
                        if num_format:
                            properties["num_format"] = num_format
                        if index % 2:
                            properties["bg_color"] = "#F1F4F7"
                        body_formats[style_key] = book.add_format(properties)
                    body = body_formats[style_key]
                    if label == "Median change (h)":
                        censored_cell = xl_rowcol_to_cell(row, frame.columns.get_loc("B50 (h)"))
                        baseline_cell = xl_rowcol_to_cell(
                            complete_row[values[0]], complete_export.columns.get_loc("B50 (h)"),
                            row_abs=True, col_abs=True,
                        )
                        sheet.write_formula(row, col, f"={censored_cell}-'Complete fits'!{baseline_cell}", body, value)
                    else:
                        sheet.write(row, col, value, body)
            sheet.autofilter(header_row, 0, header_row + len(frame), len(frame.columns) - 1)

## 9 Save the Word, PDF and Excel files

The PDF uses the contents of the generated Word report. The Excel workbook contains the fitted results, stress coefficients, censoring summary, unit records, bootstrap details, and original observations.

In [11]:
def write_pdf_with_python(word_path, pdf_path):
    """Render this report's Word paragraphs and tables with a Python PDF library."""
    from xml.sax.saxutils import escape, quoteattr

    if find_spec("reportlab") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "reportlab"], check=True)

    from docx.table import Table as WordTable
    from docx.text.paragraph import Paragraph as WordParagraph
    from docx.text.run import Run
    import reportlab
    from reportlab.lib import colors
    from reportlab.lib.enums import TA_CENTER, TA_LEFT, TA_RIGHT
    from reportlab.lib.styles import ParagraphStyle
    from reportlab.pdfbase import pdfmetrics
    from reportlab.pdfbase.ttfonts import TTFont
    from reportlab.platypus import (
        BaseDocTemplate, Frame, PageBreak, PageTemplate, Paragraph, Table, TableStyle,
    )

    # Embed fonts so character spacing is consistent across PDF viewers.
    font_pairs = [
        (Path(os.environ.get("WINDIR", "C:/Windows")) / "Fonts", "times.ttf", "timesbd.ttf"),
        (Path("/System/Library/Fonts/Supplemental"), "Times New Roman.ttf", "Times New Roman Bold.ttf"),
        (Path("/usr/share/fonts/truetype/liberation2"), "LiberationSerif-Regular.ttf", "LiberationSerif-Bold.ttf"),
        (Path("/usr/share/fonts/truetype/liberation"), "LiberationSerif-Regular.ttf", "LiberationSerif-Bold.ttf"),
        (Path("/usr/share/fonts/truetype/dejavu"), "DejaVuSerif.ttf", "DejaVuSerif-Bold.ttf"),
        (Path(reportlab.__file__).parent / "fonts", "Vera.ttf", "VeraBd.ttf"),
    ]
    for folder, regular, bold in font_pairs:
        if (folder / regular).is_file() and (folder / bold).is_file():
            pdfmetrics.registerFont(TTFont("ReportRegular", str(folder / regular)))
            pdfmetrics.registerFont(TTFont("ReportBold", str(folder / bold)))
            break
    else:
        raise RuntimeError("The PDF renderer could not find its bundled fonts.")
    pdfmetrics.registerFontFamily(
        "ReportRegular", normal="ReportRegular", bold="ReportBold",
        italic="ReportRegular", boldItalic="ReportBold",
    )

    doc = Document(word_path)
    page = doc.sections[0]
    pdf = BaseDocTemplate(
        str(pdf_path), title=doc.core_properties.title, author=doc.core_properties.author,
        pagesize=(page.page_width.pt, page.page_height.pt),
        leftMargin=page.left_margin.pt, rightMargin=page.right_margin.pt,
        topMargin=page.top_margin.pt, bottomMargin=page.bottom_margin.pt,
    )
    frame = Frame(
        pdf.leftMargin, pdf.bottomMargin, pdf.width, pdf.height,
        leftPadding=0, rightPadding=0, topPadding=0, bottomPadding=0,
        id="report-body",
    )

    def draw_page_number(canvas, template):
        canvas.saveState()
        canvas.setFont("ReportRegular", 11.5)
        canvas.drawRightString(
            page.page_width.pt - page.right_margin.pt, page.footer_distance.pt,
            str(template.page),
        )
        canvas.restoreState()

    pdf.addPageTemplates(PageTemplate(id="report", frames=[frame], onPage=draw_page_number))
    alignments = {
        WD_ALIGN_PARAGRAPH.LEFT: TA_LEFT,
        WD_ALIGN_PARAGRAPH.CENTER: TA_CENTER,
        WD_ALIGN_PARAGRAPH.RIGHT: TA_RIGHT,
    }

    def paragraph_for_pdf(paragraph):
        style = paragraph.style
        fmt = style.paragraph_format
        size = style.font.size.pt
        if style.name == "Table Entry":
            size = next((run.font.size.pt for run in paragraph.runs if run.font.size), size)
        markup = []
        for element in paragraph._p:
            if element.tag == qn("w:r"):
                run = Run(element, paragraph)
                text = escape(run.text.replace("−", "-")).replace("\n", "<br/>")
                for digit, subscript in enumerate("₀₁₂"):
                    text = text.replace(subscript, f"<sub>{digit}</sub>")
                markup.append(f"<b>{text}</b>" if run.bold else text)
            elif element.tag == qn("w:hyperlink"):
                relationship = element.get(qn("r:id"))
                url = paragraph.part.rels[relationship].target_ref
                text = escape("".join(node.text or "" for node in element.iter(qn("w:t"))))
                markup.append(f'<link href={quoteattr(url)} underline="1">{text}</link>')

        before = paragraph.paragraph_format.space_before
        before = fmt.space_before if before is None else before
        table_entry = style.name == "Table Entry"
        pdf_style = ParagraphStyle(
            style.name, fontName="ReportBold" if style.font.bold else "ReportRegular",
            fontSize=size, leading=size * (1.17 if table_entry else 1.25),
            spaceBefore=before.pt if before is not None else 0,
            spaceAfter=fmt.space_after.pt if fmt.space_after is not None else 0,
            leftIndent=fmt.left_indent.pt if fmt.left_indent is not None else 0,
            firstLineIndent=fmt.first_line_indent.pt if fmt.first_line_indent is not None else 0,
            alignment=alignments.get(paragraph.alignment, TA_LEFT),
            keepWithNext=bool(fmt.keep_with_next),
        )
        return Paragraph("".join(markup), pdf_style)

    def table_for_pdf(table):
        rows = [[paragraph_for_pdf(cell.paragraphs[0]) for cell in row.cells] for row in table.rows]
        preview = Table(rows, colWidths=[column.width.pt for column in table.columns], repeatRows=1)
        preview.hAlign = "LEFT"
        preview.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8E8E8")),
            ("GRID", (0, 0), (-1, -1), 0.35, colors.HexColor("#D9D9D9")),
            ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3.05),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3.05),
            ("TOPPADDING", (0, 0), (-1, -1), 1.95),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 1.95),
        ]))
        return preview

    story = []
    for element in doc.element.body:
        if element.tag == qn("w:p"):
            paragraph = WordParagraph(element, doc)
            if paragraph.paragraph_format.page_break_before:
                story.append(PageBreak())
            story.append(paragraph_for_pdf(paragraph))
        elif element.tag == qn("w:tbl"):
            story.append(table_for_pdf(WordTable(element, doc)))
    pdf.build(story)

In [12]:
def find_libreoffice():
    custom_path = os.environ.get("LIBREOFFICE_BIN")
    if custom_path:
        program = shutil.which(custom_path)
        if program:
            return program

    on_path = shutil.which("soffice") or shutil.which("libreoffice")
    if on_path:
        return on_path

    if sys.platform == "win32":
        for variable in ("ProgramFiles", "ProgramFiles(x86)"):
            base = os.environ.get(variable)
            if base:
                program = Path(base) / "LibreOffice" / "program" / "soffice.exe"
                if program.is_file():
                    return str(program)
    elif sys.platform == "darwin":
        program = Path("/Applications/LibreOffice.app/Contents/MacOS/soffice")
        if program.is_file():
            return str(program)

    return None


def is_pdf(path):
    if not path.is_file():
        return False
    with path.open("rb") as file:
        return file.read(5) == b"%PDF-"


def write_pdf_from_word(word_path):
    word_path = word_path.resolve()
    pdf_path = word_path.with_suffix(".pdf")
    libreoffice = find_libreoffice()
    if libreoffice:
        try:
            with TemporaryDirectory(prefix="phc_aphc_lo_") as profile:
                subprocess.run(
                    [
                        libreoffice, f"-env:UserInstallation={Path(profile).resolve().as_uri()}",
                        "--headless", "--convert-to", "pdf:writer_pdf_Export",
                        "--outdir", str(word_path.parent), str(word_path),
                    ],
                    capture_output=True, check=True, timeout=60,
                )
            if is_pdf(pdf_path):
                return pdf_path
        except (OSError, subprocess.SubprocessError):
            pass

    write_pdf_with_python(word_path, pdf_path)
    if not is_pdf(pdf_path):
        raise RuntimeError("The report did not export to a readable PDF.")
    return pdf_path


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Finish all three exports in one staging directory before publishing them.
with TemporaryDirectory(dir=OUTPUT_DIR) as temporary:
    staging = Path(temporary)
    word_path = staging / f"{REPORT_FILENAME}.docx"
    xlsx_path = staging / WORKBOOK_FILENAME
    write_word_report(word_path)
    write_excel_results(xlsx_path)
    pdf_path = write_pdf_from_word(word_path)
    exported_files = []
    for path in (word_path, pdf_path, xlsx_path):
        destination = OUTPUT_DIR / path.name
        path.replace(destination)
        exported_files.append(destination)

for path in exported_files:
    print(path)

observed_data_results\PHC_APHC_Observed_Data_Methods_and_Results.docx
observed_data_results\PHC_APHC_Observed_Data_Methods_and_Results.pdf
observed_data_results\PHC_APHC_Observed_Data_Results.xlsx
